In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [5]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import smtplib, secrets
from email.message import EmailMessage
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# Dark navy theme for the refreshed UI
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="dark"\nprimaryColor="#3b82f6"\nbackgroundColor="#020817"\nsecondaryBackgroundColor="#0b1730"\ntextColor="#eaf2ff"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#020817", "bg_sidebar": "#071329", "bg_card": "#0b1730", "bg_card_alt": "#102448",
    "text_main": "#eaf2ff", "text_heading": "#f8fbff", "text_muted": "#a6bad8",
    "accent": "#3478f6", "accent_hover": "#4f8cff", "accent_text": "#ffffff",
    "border": "#1c3b66", "border_light": "#12345c", "success": "#22c55e", "danger": "#fb7185"
}
JWT_SECRET = os.environ.get("JWT_SECRET")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")
if not JWT_SECRET:
  st.error("JWT_SECRET is not configured.")
  st.stop()
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD")
if not ADMIN_PASSWORD:
  st.error("ADMIN_PASSWORD is not configured.")
  st.stop()

# Dark futuristic glass UI styling (visual-only changes)
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Manrope:wght@400;500;600;700;800&display=swap');
    :root {{
        color-scheme: dark;
        --bg: {COLORS['bg_main']};
        --card: {COLORS['bg_card']};
        --line: {COLORS['border']};
        --blue: #22b8ff;
        --purple: #8b5cf6;
    }}
    html, body, .stApp {{
        background:
            radial-gradient(ellipse at 15% 5%, rgba(21, 91, 190, .22), transparent 35%),
            radial-gradient(ellipse at 90% 85%, rgba(111, 45, 210, .16), transparent 32%),
            {COLORS['bg_main']} !important;
        color: {COLORS['text_main']} !important;
        font-family: 'Inter', sans-serif !important;
    }}
    header {{ background: transparent !important; }}
    footer, div[data-testid="stDecoration"] {{ display: none !important; }}
    .block-container {{ max-width: 1120px; padding: 2rem 1.5rem 3rem !important; }}
    h1, h2, h3, h4 {{
        color: {COLORS['text_heading']} !important;
        font-family: 'Manrope', sans-serif !important;
        letter-spacing: -.025em;
    }}
    p, label, .stMarkdown {{ color: {COLORS['text_main']}; }}
    label p {{ color: #dceaff !important; font-weight: 500 !important; }}
    [data-testid="stTextInput"] input, [data-testid="stTextArea"] textarea,
    [data-testid="stNumberInput"] input, div[data-baseweb="select"] > div {{
        background: rgba(10, 28, 55, .9) !important;
        color: #f3f7ff !important;
        border: 1px solid #24466f !important;
        border-radius: 12px !important;
        min-height: 46px !important;
        box-shadow: inset 0 1px 0 rgba(255,255,255,.025) !important;
    }}
    [data-testid="stTextInput"] input:focus, [data-testid="stTextArea"] textarea:focus {{
        border-color: #22b8ff !important;
        box-shadow: 0 0 0 2px rgba(34,184,255,.14) !important;
    }}
    input::placeholder, textarea::placeholder {{ color: #8099bc !important; }}
    div[data-testid="stButton"] button {{
        color: white !important;
        background: linear-gradient(100deg, #16b9f5 0%, #4267f5 52%, #9654f5 100%) !important;
        border: 0 !important;
        border-radius: 999px !important;
        min-height: 46px !important;
        font-weight: 700 !important;
        box-shadow: 0 8px 24px rgba(48, 100, 240, .18) !important;
        transition: transform .18s ease, filter .18s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        filter: brightness(1.12) !important;
        transform: translateY(-1px) !important;
        border: 0 !important;
    }}
    div[data-testid="stButton"] button:focus {{ box-shadow: 0 0 0 2px rgba(34,184,255,.35) !important; }}
    section[data-testid="stSidebar"] {{
        background: linear-gradient(180deg, #071329, #050d1d) !important;
        border-right: 1px solid #17365d !important;
    }}
    .pn-card {{
        background: linear-gradient(145deg, rgba(14, 34, 65, .94), rgba(6, 18, 38, .96));
        border: 1px solid rgba(61, 126, 197, .42);
        border-radius: 18px;
        padding: 24px;
        box-shadow: 0 14px 38px rgba(0,0,0,.20), inset 0 1px 0 rgba(255,255,255,.035);
    }}
    div[data-testid="stAlert"] {{ border-radius: 12px !important; }}
    div[data-testid="stRadio"] label {{ color: #eaf2ff !important; }}
    hr {{ border-color: #1a3559 !important; }}
    @media (max-width: 700px) {{
        .block-container {{ padding: 1.2rem .8rem 2rem !important; }}
        .pn-card {{ padding: 18px; }}
    }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False
def send_otp_email(receiver_email, otp):
  msg = EmailMessage()
  msg["Subject"] = "Infosys Portal - Password Reset OTP"
  msg["From"] = EMAIL_ADDRESS
  msg["To"] = receiver_email
  msg.set_content(
      f"Your password reset OTP is: {otp}\n"
      "This code is valid for 5 minutes. Do not share it with anyone."
  )
  print("Email address configured:", bool(EMAIL_ADDRESS))
  print("Email password configured:", bool(EMAIL_PASSWORD))

  with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
    server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    server.send_message(msg)
with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("otp_code", None), ("otp_expiry", None), ("otp_attempts", 0)]:
    if k not in st.session_state: st.session_state[k] = v
if "admin_logged_in" not in st.session_state:
    st.session_state.admin_logged_in = False

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.3rem 0 1.2rem;">
        <div style="display:inline-flex;align-items:center;justify-content:center;width:64px;height:64px;
                    border-radius:22px;background:linear-gradient(135deg,#18bdf5,#5368f6 55%,#9654f5);
                    box-shadow:0 10px 32px rgba(53,112,245,.28);font-size:30px;margin-bottom:14px;">🔐</div>
        <h1 style="font-size:2rem !important;margin:0;">{title}</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:8px 0 0;">{sub}</p>
    </div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            login_type = st.radio(
                "Login as",
                ["User", "Admin"],
                horizontal=True
            )
            auth_header("Sign in to your account")
            email = st.text_input("Username or Email", placeholder="username or you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
              if not email or not pwd:
                st.error("Please fill in all fields.")
              elif login_type == "Admin":

                if email == ADMIN_USERNAME and pwd == ADMIN_PASSWORD:
                  st.session_state.admin_logged_in = True
                  st.session_state.token = make_jwt("admin@infosys.local")
                  navigate("Dashboard")
                else:
                  st.error("Invalid admin username or password.")
              else:
                with get_db() as c:
                  r = c.execute(
                      "SELECT password_hash, email FROM users WHERE email=? OR LOWER(username)=?",
                      (email, email)
                  ).fetchone()

                if r and check_txt(pwd, r[0]):
                  st.session_state.admin_logged_in = False
                  st.session_state.token = make_jwt(r[1])
                  navigate("Dashboard")
                else:
                  st.error("Invalid email or password.")



            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)


            if st.button("Create Account →", use_container_width=True):
                import re

                if not uname.strip():
                    st.error("Please enter your username.")
                elif not email:
                    st.error("Please enter your email address.")
                elif not re.fullmatch(r"[A-Za-z0-9._%+-]{2,}@[A-Za-z]{2,}(?:\.[A-Za-z]{2,})+", email):
                    st.error("Please enter a valid email (e.g. ab@cd.ef).")
                elif not pwd:
                    st.error("Please enter a password.")
                elif len(pwd) < 8:
                    st.error("Password must be at least 8 characters.")
                elif not re.search(r"[A-Z]", pwd):
                    st.error("Password must contain at least one uppercase letter.")
                elif not re.search(r"[a-z]", pwd):
                    st.error("Password must contain at least one lowercase letter.")
                elif not re.search(r"\d", pwd):
                    st.error("Password must contain at least one number.")
                elif not re.search(r"[^A-Za-z0-9]", pwd):
                    st.error("Password must contain at least one special character.")
                elif not confirm_pwd:
                    st.error("Please confirm your password.")
                elif pwd != confirm_pwd:
                    st.error("Passwords do not match.")
                elif not sa.strip():
                    st.error("Please enter your security answer.")
                else:
                  with get_db() as c:
                      dup_user = c.execute("SELECT 1 FROM users WHERE LOWER(username)=?", (uname.strip().lower(),)).fetchone()
                      dup_email = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                  if dup_user:
                      st.error("This username is already taken. Please choose another.")
                  elif dup_email:
                      st.error("This email is already registered. Please sign in instead.")
                  else:
                      try:
                          with get_db() as c:
                              c.execute(
                                  "INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                                  (uname.strip(), email, hash_txt(pwd), sq, hash_txt(sa.lower().strip()))
                              )
                          st.success("Account created successfully! Please sign in.")
                          time.sleep(1)
                          navigate("Login")
                      except sqlite3.IntegrityError:
                          st.error("Username or email already registered.")


            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if not st.session_state.reset_email:
                email = st.text_input("Registered username or email", placeholder="username or you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    else:
                        with get_db() as c:
                            r = c.execute(
                                "SELECT security_question, email FROM users WHERE email=? OR LOWER(username)=?",
                                (email, email)
                            ).fetchone()
                        if r:
                            st.session_state.reset_email = r[1]
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    else:
                        with get_db() as c:
                            r = c.execute(
                                "SELECT email FROM users WHERE email=? OR LOWER(username)=?",
                                (email, email)
                            ).fetchone()
                        if not r:
                            st.error("Email not found. Please check your registered mail.")
                        else:
                            otp = str(secrets.randbelow(900000) + 100000)
                            try:
                                send_otp_email(r[0], otp)
                                st.session_state.reset_email = r[0]
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp_verified = False
                                st.session_state.otp_code = otp
                                st.session_state.otp_expiry = time.time() + 300
                                st.session_state.otp_attempts = 0
                                st.session_state.pop("reset_otp_input", None)
                                st.rerun()
                            except Exception as e:
                                st.error("Could not send OTP email. Please try again later.")

            else:
              if st.session_state.get("reset_mode") == "sq":
                  st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                  ans = st.text_input("Your answer").lower().strip()
                  npw = st.text_input("New password (min 8 chars)", type="password")
                  confirm_npw = st.text_input("Confirm new password", type="password")

                  if st.button("Reset Password →", use_container_width=True):
                    import re

                    if not ans:
                      st.error("Please enter your security answer.")
                    elif not npw:
                      st.error("Please enter a new password.")
                    elif len(npw) < 8:
                      st.error("Password must be at least 8 characters.")
                    elif not re.search(r"[A-Z]", npw):
                      st.error("Password must contain at least one uppercase letter.")
                    elif not re.search(r"[a-z]", npw):
                      st.error("Password must contain at least one lowercase letter.")
                    elif not re.search(r"\d", npw):
                      st.error("Password must contain at least one number.")
                    elif not re.search(r"[^A-Za-z0-9]", npw):
                      st.error("Password must contain at least one special character.")
                    elif not confirm_npw:
                      st.error("Please confirm your new password.")
                    elif npw != confirm_npw:
                      st.error("Passwords do not match.")
                    else:
                      with get_db() as c:
                        r = c.execute(
                            "SELECT security_answer_hash FROM users WHERE email=?",
                            (st.session_state.reset_email,)
                        ).fetchone()
                      if r and check_txt(ans, r[0]):
                        with get_db() as c:
                          c.execute(
                              "UPDATE users SET password_hash=? WHERE email=?",
                              (hash_txt(npw), st.session_state.reset_email)
                          )
                        st.session_state.reset_email = None
                        st.session_state.reset_mode = None
                        st.success("Password updated successfully!")
                        time.sleep(1)
                        navigate("Login")
                      else:
                        st.error("Incorrect security answer.")


              elif st.session_state.get("reset_mode") == "otp_locked":
                  st.error("🚫 Too many incorrect OTP attempts!")
                  st.warning("For your security, this OTP has been blocked after 3 incorrect attempts. Press Cancel to return to Login and request a new OTP.")

              elif st.session_state.get("reset_mode") == "otp":
                  # STEP 1: Verify OTP only
                  if not st.session_state.get("otp_verified", False):
                      st.subheader("Step 1: Verify OTP")
                      st.info("Enter the 6-digit OTP sent to your email.")

                      entered_otp = st.text_input("OTP", max_chars=6, key="reset_otp_input")

                      if st.button("Verify OTP", use_container_width=True):
                          if not entered_otp:
                              st.error("Please enter the OTP.")
                          elif time.time() > st.session_state.otp_expiry:
                              st.error("OTP expired. Please cancel and request a new OTP.")
                          elif entered_otp != st.session_state.otp_code:
                              st.session_state.otp_attempts = st.session_state.get("otp_attempts", 0) + 1
                              remaining = 3 - st.session_state.otp_attempts
                              if remaining <= 0:
                                  st.session_state.reset_mode = "otp_locked"
                                  st.session_state.otp_code = None
                                  st.session_state.otp_verified = False
                                  st.session_state.pop("reset_otp_input", None)
                                  st.rerun()
                              else:
                                  st.error(f"Incorrect OTP. You have {remaining} attempt(s) remaining.")
                          else:
                              st.session_state.otp_verified = True
                              st.rerun()

                  # STEP 2: Show password fields only after OTP verification
                  else:
                      import re

                      st.subheader("Step 2: Create a new password")
                      st.success("OTP verified successfully.")
                      npw = st.text_input("New password (min 8 chars)", type="password", key="new_reset_password")
                      confirm_npw = st.text_input("Confirm new password", type="password", key="confirm_reset_password")

                      if st.button("Reset Password", use_container_width=True):
                          if not npw:
                              st.error("Please enter a new password.")
                          elif len(npw) < 8:
                              st.error("Password must be at least 8 characters.")
                          elif not re.search(r"[A-Z]", npw):
                              st.error("Password must contain at least one uppercase letter.")
                          elif not re.search(r"[a-z]", npw):
                              st.error("Password must contain at least one lowercase letter.")
                          elif not re.search(r"\d", npw):
                              st.error("Password must contain at least one number.")
                          elif not re.search(r"[^A-Za-z0-9]", npw):
                              st.error("Password must contain at least one special character.")
                          elif not confirm_npw:
                              st.error("Please confirm your new password.")
                          elif npw != confirm_npw:
                              st.error("Passwords do not match.")
                          else:
                              with get_db() as c:
                                  c.execute(
                                      "UPDATE users SET password_hash=? WHERE email=?",
                                      (hash_txt(npw), st.session_state.reset_email)
                                  )

                              st.session_state.reset_email = None
                              st.session_state.reset_mode = None
                              st.session_state.otp_code = None
                              st.session_state.otp_expiry = None
                              st.session_state.otp_verified = False
                              st.session_state.otp_attempts = 0
                              st.session_state.pop("reset_otp_input", None)
                              st.session_state.pop("new_reset_password", None)
                              st.session_state.pop("confirm_reset_password", None)

                              st.success("Password reset successfully! Please log in.")
                              time.sleep(1)
                              navigate("Login")

              st.markdown("<br>", unsafe_allow_html=True)
              if st.button("← Cancel", use_container_width=True):
                  st.session_state.reset_email = None
                  st.session_state.reset_mode = None
                  st.session_state.otp_code = None
                  st.session_state.otp_expiry = None
                  st.session_state.otp_verified = False
                  st.session_state.otp_attempts = 0
                  st.session_state.pop("reset_otp_input", None)
                  st.session_state.pop("new_reset_password", None)
                  st.session_state.pop("confirm_reset_password", None)
                  navigate("Login")


# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    is_admin = st.session_state.get("admin_logged_in", False)

    if is_admin:
      email = "admin@infosys.local"
      uname = "Administrator"
    else:
      email = payload["email"]
      with get_db() as c:
        user_row = c.execute(
            "SELECT username FROM users WHERE email=?",
            (email,)
        ).fetchone()
      if user_row is None:
        st.session_state.token = None
        st.session_state.admin_logged_in = False
        navigate("Login")
        st.stop()

      uname = user_row[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if is_admin else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if is_admin else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if is_admin else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.admin_logged_in = False
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if is_admin:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("Registered Users")

        with get_db() as c:
          users = c.execute(
              "SELECT username, email FROM users ORDER BY username"
          ).fetchall()

        if users:
          st.dataframe(
              [{"Username": row[0], "Email": row[1]} for row in users],
              use_container_width=True,
              hide_index=True
          )
        else:
          st.info("No registered users yet.")

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.success(f"Welcome, {uname}! 👋 You are logged in as {email}.")
        st.markdown("<br>", unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [7]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve all secrets securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
JWT_SECRET = userdata.get('JWT_SECRET')
EMAIL_ADDRESS = userdata.get('EMAIL_ADDRESS')
EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD')
ADMIN_PASSWORD = userdata.get('ADMIN_PASSWORD')


# ngrok.set_auth_token(NGROK_TOKEN)
# Check that all required secrets are available
if not all([NGROK_TOKEN, JWT_SECRET, EMAIL_ADDRESS, EMAIL_PASSWORD, ADMIN_PASSWORD]):
    raise RuntimeError(
        "Missing secrets. Check Colab Secrets and enable notebook access."
    )

# Pass the app secrets to the Streamlit process
os.environ["JWT_SECRET"] = JWT_SECRET
os.environ["EMAIL_ADDRESS"] = EMAIL_ADDRESS
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD
os.environ["ADMIN_PASSWORD"] = ADMIN_PASSWORD

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://playoff-monument-target.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.

🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑
Received Ctrl+C / Stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.


In [8]:
from pyngrok import ngrok

# Stop existing ngrok tunnels
ngrok.kill()



In [9]:
%%writefile requirements.txt
streamlit
PyJWT
bcrypt
pyngrok
plotly
streamlit-option-menu

Writing requirements.txt
